In [1]:
from pathlib import Path
import csv
import json
import shutil

LAST_NAME = "Rojas"
STUDENT_ID = "TUPM-26-1244"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_CLASS = (SEED_DIGIT % 5) + 1

print("Seed:", SEED_DIGIT)
print("Target OS:", TARGET_OS)
print("Target behavior class:", TARGET_CLASS)

Seed: 4
Target OS: Android
Target behavior class: 5


In [4]:
WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "expanded_user_behavior_dataset.csv"
print("Dataset found:", dataset_path.exists())

Dataset found: True


In [5]:
EXPORT_FIELDS = [
    "User ID", "Device Model", "Operating System",
    "Battery Drain (mAh/day)", "Data Usage (MB/day)",
]

total_records = 0
os_matches = 0
selected = []
battery_total = 0.0
data_total = 0.0
invalid_records = 0

with open(dataset_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    for row in reader:
        total_records += 1

        if (row.get("Operating System") or "").strip() != TARGET_OS:
            continue
        os_matches += 1

        try:
            behavior_class = int(row["User Behavior Class"])
            battery = float(row["Battery Drain (mAh/day)"])
            data_usage = float(row["Data Usage (MB/day)"])
        except (KeyError, ValueError, TypeError):
            invalid_records += 1
            continue

        if behavior_class == TARGET_CLASS:
            selected.append({field: row[field] for field in EXPORT_FIELDS})
            battery_total += battery
            data_total += data_usage

selected_count = len(selected)
avg_battery = battery_total / selected_count if selected_count else 0
avg_data = data_total / selected_count if selected_count else 0

print("Total records in dataset:", total_records)
print("Records matching operating system:", os_matches)
print("Records in target behavior class:", selected_count)
print("Invalid records skipped:", invalid_records)
print("Average battery drain (mAh/day):", round(avg_battery, 3))
print("Average data usage (MB/day):", round(avg_data, 3))
if selected_count == 0:
    print("No matching records found.")

Total records in dataset: 1000
Records matching operating system: 801
Records in target behavior class: 152
Invalid records skipped: 0
Average battery drain (mAh/day): 2711.678
Average data usage (MB/day): 1996.513


In [6]:
csv_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_selected.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"

with open(csv_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=EXPORT_FIELDS)
    writer.writeheader()
    writer.writerows(selected)

with open(csv_path, "r", encoding="utf-8", newline="") as file:
    exported_rows = sum(1 for _ in csv.DictReader(file))

count_ok = exported_rows == selected_count

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": TARGET_OS,
    "behavior_class": TARGET_CLASS,
    "total_records": total_records,
    "os_matches": os_matches,
    "selected_records": selected_count,
    "average_battery_drain_mAh_day": round(avg_battery, 3),
    "average_data_usage_MB_day": round(avg_data, 3),
    "exported_row_count": exported_rows,
    "count_verified": count_ok,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("Exported rows:", exported_rows, "| Selected:", selected_count)
print("Count verification:", "VERIFIED" if count_ok else "MISMATCH")

Exported rows: 152 | Selected: 152
Count verification: VERIFIED


In [7]:
backup_path = BACKUP_DIR / csv_path.name
shutil.copy2(csv_path, backup_path)
backup_size = backup_path.stat().st_size

print("Backup exists:", backup_path.exists())
print("Backup file:", backup_path.name)
print("Backup size:", backup_size, "bytes")

Backup exists: True
Backup file: Rojas_TUPM-26-1244_mobile_selected.csv
Backup size: 5695 bytes


In [8]:
print("=== FINAL OUTPUT: MOBILE DEVICE PROFILING ===")
print("OS and behavior class:", TARGET_OS, "/ Class", TARGET_CLASS)
print("Total records in dataset:", total_records)
print("Records matching OS:", os_matches)
print("Records in target class:", selected_count)
print("Average battery drain (mAh/day):", round(avg_battery, 3))
print("Average data usage (MB/day):", round(avg_data, 3))
print("Count verification:", "VERIFIED" if count_ok else "MISMATCH")
print("Exported CSV file:", csv_path.name)
print("JSON summary file:", json_path.name)
print("Backup file and size:", backup_path.name, backup_size, "bytes")

=== FINAL OUTPUT: MOBILE DEVICE PROFILING ===
OS and behavior class: Android / Class 5
Total records in dataset: 1000
Records matching OS: 801
Records in target class: 152
Average battery drain (mAh/day): 2711.678
Average data usage (MB/day): 1996.513
Count verification: VERIFIED
Exported CSV file: Rojas_TUPM-26-1244_mobile_selected.csv
JSON summary file: Rojas_TUPM-26-1244_mobile_summary.json
Backup file and size: Rojas_TUPM-26-1244_mobile_selected.csv 5695 bytes
